In [31]:
import spacy
import pandas as pd
from spacy.tokens import DocBin
from spacy.language import Language
from spacy import displacy


In [43]:
training_data = [
    ("3 units from Computer Science 219, 233, 235, Computer Engineering 335, 339 or Software Engineering for Engineers 337.",
     [(14, 33, "COURSE"), (36, 38, "COURSE"), (41, 43, "COURSE"), (46, 69, "COURSE"), (72, 74, "COURSE"), (79, 116, "COURSE")],
     [(14, 29), (46, 66), (79, 113)]),

    ("Computer Science 319 or 331; and Computer Science 355 or Computer Engineering 369.",
     [(0, 20, "COURSE"), (25, 27, "COURSE"), (34, 53, "COURSE"), (58, 81, "COURSE")], 
     [(0, 17), (34, 50), (58, 78)]),

    ("Communication and Media Studies 369.", [(0, 33, "COURSE")], [(0, 30)]),

    ("Economics 387 and Mathematics 211.", [(0, 13, "COURSE"), (19, 33, "COURSE")], []),

    ("Economics 201 and 203; and 3 units from Mathematics 249, 265 or 275.",
     [(0, 13, "COURSE"), (19, 21, "COURSE"), (41, 55, "COURSE"), (58, 60, "COURSE"), (65, 67, "COURSE")], 
     []),
    
    ("Credit for Computer Science 503 and either 502 or Software Engineering for Engineers 599 will not be allowed.", 
     [(12, 31, "COURSE"), (44, 46, "COURSE"), (51, 88, "COURSE")], 
     [(12, 29), (51, 85)]),
    
    ("3 units from Computer Science 319, 331 or Software Engineering for Engineers 338; and 3 units from Software Engineering 300, 301 or Software Engineering for Engineers 480; and admission to the Schulich School of Engineering.", 
     [(14, 33, "COURSE"), (36, 38, "COURSE"), (43, 80, "COURSE"), (100, 123, "COURSE"), (126, 128, "COURSE"), (133, 170, "COURSE"), (194, 223, "FACULTY")], 
     [(14, 29), (43, 76), (100, 119), (133, 166), (194, 223)]),
    
    ("3 units from Philosophy 309, 311, 333, 359, Religious Studies 363; and 3 units in a course labelled Philosophy.", 
     [(14, 26, "COURSE"), (30, 32, "COURSE"), (35, 37, "COURSE"), (40, 42, "COURSE"), (45, 65, "COURSE"), (101, 110, "COURSE")], 
     [(14, 23), (45, 61), (101, 110)]),
]


In [47]:
@Language.component("sentence_segmenter")
def sentence_segmenter(doc):
    SPLITTERS = [".", ";"]
    for token in doc[:-1]:
        if token.text in SPLITTERS:
            doc[token.i + 1].is_sent_start = True
    return doc


In [48]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("sentence_segmenter", before="parser")

db = DocBin()

for text, annotations, retoken in training_data:
    # print(text)
    
    doc = nlp(text)
    ents = []
    for start, end, label in annotations:
        span = doc.char_span(start, end, label=label, alignment_mode="expand")
        ents.append(span)
    doc.set_ents(ents)

    for start, end in retoken:
        with doc.retokenize() as retokenizer:
            span = doc.char_span(start, end, label="", alignment_mode="expand")
            retokenizer.merge(doc[span.start:span.end])
        
    assert doc.has_annotation("SENT_START")
    for sent in doc.sents:
        print(sent.text)


    db.add(doc)
    
    displacy.render(doc, style="ent", jupyter=True)
    # displacy.render(doc, style="span", jupyter=True)
    displacy.render(doc, style="dep", jupyter=True)


3 units from Computer Science 219, 233, 235, Computer Engineering 335, 339 or Software Engineering for Engineers 337.


Computer Science 319 or 331;
and Computer Science 355 or Computer Engineering 369.


Communication and Media Studies 369.


Economics 387 and Mathematics 211.


Economics 201 and 203;
and 3 units from Mathematics 249, 265 or 275.


Credit for Computer Science 503 and either 502 or Software Engineering for Engineers 599 will not be allowed.


3 units from Computer Science 319, 331 or Software Engineering for Engineers 338;
and 3 units from Software Engineering 300, 301 or Software Engineering for Engineers 480;
and admission to the Schulich School of Engineering.


3 units from Philosophy 309, 311, 333, 359, Religious Studies 363;
and 3 units in a course labelled Philosophy.


In [49]:
print(nlp.pipe_names)


['tok2vec', 'tagger', 'sentence_segmenter', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [46]:
db.to_disk("./training-data.spacy")
